# 方案五：管道式 ACOS（Stage1 对抽取 + Stage2 交叉编码分类）复现

将任务解耦成两个职责单一的 BERT 模型：

```
评论 ──Stage1──▶ 若干 (Aspect, Opinion) 对 ──Stage2──▶ 每对补上 (Category, Polarity) ──▶ 四元组
```

| 阶段 | 模型 | dev 指标（可本地复算） |
| --- | --- | --- |
| Stage1 | BERT-CRF span + span MLP + 双仿射配对（隐式 A/O 用可学习哨兵） | 配对 F1 **0.8038** |
| Stage2 | 交叉编码器 `[CLS] review [SEP] 方面词:a 观点词:o [SEP]` | Category macro-F1 **0.9036**，Polarity macro-F1 **0.8619**，给定 gold 对的四元组 acc **0.9360** |
| 端到端 | Stage1→Stage2 | dev 四元组 F1 **约 0.7587**；测试集 **0.7651**（排行榜） |

> 训练使用 GPU；脚本产物路径全部写入 `src/config.py` 规定的位置（项目根目录 `data/pipeline/`、`outputs/`、`config/` 下）。

## Step 0. 环境与依赖

**国内机器先配清华镜像**可加速 5~10 倍：

```bash
pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple
pip config set global.trusted-host pypi.tuna.tsinghua.edu.cn
```

然后安装（本 Notebook 使用 GPU）：

```bash
pip install torch --index-url https://download.pytorch.org/whl/cu121
pip install -r requirements.txt
export HF_ENDPOINT=https://hf-mirror.com   # HuggingFace 直连慢时
```

In [ ]:
import importlib.util, os

need = {"torch": "torch", "transformers": "transformers", "pytorch-crf": "torchcrf"}
missing = [pkg for pkg, mod in need.items() if importlib.util.find_spec(mod) is None]
print("依赖检查:", "全部就绪" if not missing else f"缺少 {missing}")
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [ ]:
# 统一入口：定位项目根目录（notebook/ 的上一级），把 src/ 加入 import 路径
# 脚本产物路径保持原样（config.py 中的 BASE_DIR/outputs, BASE_DIR/data/...）
import os, sys, subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 脚本产物写入的根目录（与 src/config.py 保持一致）
ARTIFACT_DIR = PROJECT_ROOT

# 默认使用 0 号 GPU；多卡机器可改成 "1" 等
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

print("项目根目录:", PROJECT_ROOT)
print("产物根目录:", ARTIFACT_DIR)

def run(cmd):
    """用当前 Python 解释器执行命令，工作目录为 PROJECT_ROOT。"""
    print(">>", " ".join(map(str, cmd)))
    subprocess.run([str(c) for c in cmd], check=True, env=os.environ.copy())

In [ ]:
# GPU 环境检查：本实验要求使用 CUDA
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch

assert torch.cuda.is_available(), "未检测到 CUDA GPU：请在有 NVIDIA GPU 的实例上运行本 Notebook"
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name)
print("显存: %.1f GB" % (props.total_memory / 1e9))
print("CUDA:", torch.version.cuda, "| PyTorch:", torch.__version__)
print("bf16 支持:", torch.cuda.is_bf16_supported())
torch.backends.cudnn.benchmark = True

## Step 1. 数据校验（注意必须用原文）

Stage1 依赖标签里的**字符偏移**，必须读取 `data/raw/TRAIN/Train_reviews.csv` 原文。`outputs/processed/Train_reviews_processed.csv` 压缩了连续空格（3229 条中 78 条受影响），用它会导致 span 错位（如“物流/很快”切成“流很/快”）。

本方案**不需要**先运行预处理脚本。

In [ ]:
import pandas as pd
from pathlib import Path

paths = [
    "data/raw/TRAIN/Train_reviews.csv",
    "data/raw/TRAIN/Train_labels.csv",
    "data/raw/TEST/Test_reviews.csv",
]
for p in paths:
    print(f"{p:45s} 存在: {Path(p).exists()}")

train = pd.read_csv("data/raw/TRAIN/Train_reviews.csv", encoding="utf-8")
labels = pd.read_csv("data/raw/TRAIN/Train_labels.csv", encoding="utf-8")
test = pd.read_csv("data/raw/TEST/Test_reviews.csv", encoding="utf-8")
print("训练评论:", train.shape, "| 训练标签:", labels.shape, "| 测试评论:", test.shape)
assert len(train) == 3229 and len(labels) == 6633 and len(test) == 2237, "数据规模与预期不一致，请检查原始数据"
display(train.head(3))

## Step 2. 查看两阶段超参

`config/model_pipeline.yaml` 同时管理 Stage1 与 Stage2：
- Stage1：6 epoch、max_len 256、batch 8、span 维度 200、配对 BCE 正样本权重 2、阈值搜索 [0.3, 0.4, 0.5, 0.6, 0.7]；
- Stage2：5 epoch、max_len 320、batch 12、截断逆频率类权重 cap=10、隐式占位符“无”。

In [ ]:
import yaml
cfg = yaml.safe_load(Path("config/model_pipeline.yaml").read_text(encoding="utf-8"))
print("Stage1 超参:", cfg["hyperparams"])
print("Stage1 配对:", cfg["pair"])
print("Stage2 超参:", cfg["clf"])

## Step 3. Stage1：(Aspect, Opinion) 对抽取

```bash
python src/pipeline/extract_pair.py
# 冒烟测试（只截少量样本，验证流程）：
python src/pipeline/extract_pair.py --max-train 64 --max-dev 32
```

In [ ]:
# 完整训练（使用 GPU）
run([sys.executable, "src/pipeline/extract_pair.py"])

# 冒烟测试可改为：
# run([sys.executable, "src/pipeline/extract_pair.py", "--max-train", "64", "--max-dev", "32"])

In [ ]:
text = (ARTIFACT_DIR / "outputs/logs/pipeline_pair_report.txt").read_text(encoding="utf-8")
print(text)

In [ ]:
import json
meta = json.loads((ARTIFACT_DIR / "data/pipeline/pair_meta.json").read_text(encoding="utf-8"))
print("最优配对阈值 tau:", meta["best_threshold"])
print("dev 配对指标:", meta["best_dev"])
print("span 抽取指标:", meta["span_dev"])
print("权重文件存在:", (ARTIFACT_DIR / "data/pipeline/pair_extractor.pt").exists())

## Step 4. Stage2：逐对 (Category, Polarity) 交叉编码分类

```bash
python src/pipeline/classify_pair.py
```

Stage2 与 Stage1 使用完全相同的 `seed=42` 评论级 85/15 切分，保证 dev 评论对齐。

In [ ]:
# 完整训练（使用 GPU）
run([sys.executable, "src/pipeline/classify_pair.py"])

# 冒烟测试可改为：
# run([sys.executable, "src/pipeline/classify_pair.py", "--max-train", "64", "--max-dev", "32"])

In [ ]:
text = (ARTIFACT_DIR / "outputs/logs/pipeline_clf_report.txt").read_text(encoding="utf-8")
print(text)

In [ ]:
import json
clf_meta = json.loads((ARTIFACT_DIR / "data/pipeline/pair_clf_meta.json").read_text(encoding="utf-8"))
bd = clf_meta["best_dev"]
print("dev 样本（四元组级）:", bd["n"])
print("Category macro-F1=%.4f acc=%.4f" % (bd["cat_macro_f1"], bd["cat_acc"]))
print("Polarity macro-F1=%.4f acc=%.4f" % (bd["pol_macro_f1"], bd["pol_acc"]))
print("给定 gold 对的四元组准确率=%.4f" % bd["quad_acc"])
print("\nCategory 每类明细:")
display(pd.DataFrame(bd["cat_detail"]).round(3))
print("\nPolarity 每类明细:")
display(pd.DataFrame(bd["pol_detail"]).round(3))
print("\n最易混淆类别对:")
display(pd.DataFrame(bd["top_confused"]))

## Step 5. 串联推理

```bash
python src/pipeline/predict_pipeline.py --eval-dev   # dev 端到端四元组 P/R/F1
python src/pipeline/predict_pipeline.py              # 对测试集推理，写 data/pipeline/Result.csv
```

In [ ]:
# dev 端到端评估（需要 Stage1/Stage2 两个权重都已训练完成）
run([sys.executable, "src/pipeline/predict_pipeline.py", "--eval-dev"])

In [ ]:
# 对 2237 条测试评论推理
run([sys.executable, "src/pipeline/predict_pipeline.py"])

In [ ]:
import pandas as pd
from pathlib import Path

COLS = ["id", "AspectTerms", "OpinionTerms", "Categories", "Polarities"]
CATS = {"整体", "使用体验", "功效", "价格", "物流", "气味", "包装",
        "真伪", "服务", "其他", "成分", "尺寸", "新鲜度"}
POLS = {"正面", "负面", "中性"}

def validate_result(path):
    p = Path(path)
    assert p.exists(), f"结果文件不存在: {p}"
    df = pd.read_csv(p, header=None, names=COLS, dtype=str)
    df["id"] = df["id"].str.strip().astype(int)
    test_ids = set(pd.read_csv("data/raw/TEST/Test_reviews.csv")["id"].astype(int))
    assert set(df["id"]) == test_ids, "测试集 id 覆盖不完整"
    assert set(df["Categories"].str.strip()) <= CATS, "出现非法类别"
    assert set(df["Polarities"].str.strip()) <= POLS, "出现非法极性"
    for c in COLS[1:]:
        df[c] = df[c].str.strip()
    print(f"[通过] {p}")
    print(f"  行数（四元组数）: {len(df)} | 覆盖 id: {df['id'].nunique()} | 密度: {len(df)/len(test_ids):.2f} 个/条")
    print("  类别分布:", dict(df["Categories"].value_counts()))
    print("  极性分布:", dict(df["Polarities"].value_counts()))
    return df

df = validate_result(ARTIFACT_DIR / "data/pipeline/Result.csv")
display(df.head(10))

## 结果解读

- dev 端到端四元组 F1 约 **0.7587**，测试集 **0.7651**，基本消除了零抽取；
- 输出密度约 2.1 个四元组/条，与训练集 2.05 接近；
- 误差分析显示**端到端假阳性中约 83% 在 Stage1 已经配错对**，Stage2 本身准确率很高。因此后续提分应优先优化 Stage1（调阈值、增加训练轮数、改进 span 表示）。